# Hugging Faceを使った多言語推論モデルのファインチューニング(openai Official)

著者: [Edward Beeching](https://huggingface.co/edbeeching), [Quentin Gallouédec](https://huggingface.co/qgallouedec), [Lewis Tunstall](https://huggingface.co/lewtun)

[OpenAI o3](https://openai.com/index/introducing-o3-and-o4-mini/)のような大規模推論モデルは、思考の連鎖（chain-of-thought）を生成して応答の精度と品質を向上させます。しかし、これらのモデルの多くは、他の言語で質問されても英語で推論を行います。

このノートブックでは、OpenAIのオープンウェイト推論モデル [OpenAI gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b) を複数の言語で効果的に推論できるようにファインチューニングする方法を紹介します。モデルのシステムプロンプトに新しい「推論言語」オプションを追加し、Hugging Faceの [TRL ライブラリ](https://github.com/huggingface/trl)を使って多言語推論データセットで[教師あり学習](https://huggingface.co/learn/llm-course/chapter11/1)を適用します。

以下の手順をカバーします：

1. **セットアップ:** 必要なライブラリをインストールします。
2. **データセットの準備:** ファインチューニング用のデータセットをダウンロードし、フォーマットします。
3. **モデルの準備:** ベースモデルを読み込み、メモリ効率的な手法である [LoRA](https://huggingface.co/learn/llm-course/chapter11/4) でファインチューニング用に設定します。
4. **ファインチューニング:** 多言語推論データでモデルを訓練します。
5. **推論:** ファインチューニング後のモデルを使って、異なる言語で推論応答を生成します。

最終的に、英語、スペイン語、フランス語、イタリア語、ドイツ語で思考の連鎖を生成できる多言語推論モデルが完成します。言語を混在させることも可能です。例えば、スペイン語で質問し、ドイツ語での推論を要求し、最終的な応答をスペイン語で受け取ることができます：

```txt
User:
    ¿Cuál es el capital de Australia?
Assistant reasoning:
    Okay, der Benutzer fragt nach der Hauptstadt Australiens. Ich erinnere mich, dass Canberra die Hauptstadt ist. Ich
    sollte das bestätigen. Lass mich sehen, ob es irgendwelche potenziellen Verwirrungen gibt. Der Benutzer könnte auch
    an der größten Stadt interessiert sein. Die größte Stadt ist Sydney, aber die Hauptstadt ist Canberra. Ich sollte
    das klarstellen. Vielleicht auch erwähnen, dass Canberra eine geplante Stadt ist und nicht die größte. Der Benutzer
    könnte auch nach der Geografie fragen. Vielleicht erwähne ich, dass Canberra im südwestlichen Teil der Australian
    Capital Territory liegt. Ich sollte die Antwort präzise und freundlich halten. Vielleicht auch erwähnen, dass
    Canberra oft mit Sydney verwechselt wird. Ich sollte sicherstellen, dass die Antwort klar und korrekt ist.
Assistant response:
    La capital de Australia es **Canberra**. Aunque es la ciudad más pequeña de las principales capitales del país, fue
    elegida en 1908 como la sede del gobierno federal para equilibrar la influencia entre las ciudades de Sydney y
    Melbourne. Canberra está ubicada en el Territorio de la Capital Australiana (ACT), en el este de Australia.
```

このチュートリアルが、十分に代表されていない言語で作業するAI開発者が、母国語での [`openai/gpt-oss-20b`](https://huggingface.co/openai/gpt-oss-20b) の解釈可能性を向上させるのに役立つことを願っています。

> **注意:** このノートブックは、80GBメモリを持つ単一のH100 GPUで実行するように設計されています。より小さなGPUを使用する場合は、以下のハイパーパラメータでバッチサイズとシーケンス長を削減できます。

## セットアップ

開始するために、必要なライブラリをすべてインストールしましょう。最初にPyTorchをインストールします：

In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cu128

次に、残りの依存関係をインストールします：

In [ ]:
%pip install "trl>=0.20.0" "peft>=0.17.0" "transformers>=4.55.0" trackio

最後に、以下のようにHugging Faceアカウントにログインします：

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

必要なライブラリをインストールしたので、ファインチューニングに使用するデータセットを見てみましょう。

## データセットの準備

[Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking)を使用します。これは、思考の連鎖がフランス語、スペイン語、ドイツ語などの複数の言語に翻訳された推論データセットです。このデータセットで `openai/gpt-oss-20b` をファインチューニングすることで、これらの言語で推論ステップを生成することを学習し、それらの言語を話すユーザーが推論プロセスを理解できるようになります。

<iframe
  src="https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking/embed/viewer/default/train"
  frameborder="0"
  width="100%"
  height="560px"
></iframe>



Hugging Face Hubからこのデータセットをダウンロードしましょう：

In [ ]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/Multilingual-Thinking", split="train")
dataset

これは1,000例の小さなデータセットですが、通常は広範囲な事後訓練を受けた `openai/gpt-oss-20b` のようなモデルには十分です。訓練例の1つを見てみましょう：

In [ ]:
dataset[0]

`gpt-oss` モデルは、会話構造の定義、推論出力の生成、関数呼び出しの構造化のためのHarmony応答フォーマットで訓練されました。このフォーマットはOpenAI Responses APIを模倣するように設計されており、以下の表はデータセットで使用される異なるメッセージタイプをまとめています：

|||
| :---- | :--|
| `developer` | developer メッセージは、モデルにカスタム指示を提供するために使用されます（通常 `system` ロールと呼ばれるもの）。|
| `user` | user メッセージは、モデルに入力を提供するために使用されます。|
| `assistant` | モデルによって出力され、ツール呼び出しまたはメッセージ出力のいずれかになります。出力は特定の「チャンネル」に関連付けられ、メッセージの意図を特定することもあります。|
| `analysis` | これらは、モデルが思考の連鎖に使用するメッセージです。|
| `final` | final チャンネルでタグ付けされたメッセージは、エンドユーザーに表示されることを意図したメッセージで、モデルからの応答を表します。|
| `messages` | 上記の内容を組み合わせて完全な会話を作成するメッセージのリスト。これがモデルへの入力です。|

[OpenAIのメッセージフォーマット](https://platform.openai.com/docs/api-reference/messages/object)に慣れている場合、これは非常に似ていることがわかりますが、重要な違いがあります：

> `assistant` ターンには2つの特別なフィールドが含まれています：モデルの推論プロセスを含む `thinking` フィールドと、ユーザーへの最終応答を含む `content` フィールドです。

モデルをファインチューニングするには、これらのメッセージをモデルが理解できる形式に変換する必要があります。実際には、これはモデルの[_チャットテンプレート_](https://huggingface.co/docs/transformers/chat_templating)で各メッセージをフォーマットし、結果のテキストをトークン化することで行われます。TRLライブラリはこれを自動的に行いますが、仕組みを理解するために段階的に説明しましょう。

まず、トークナイザーを読み込みます：

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

次に、トークナイザーの `apply_chat_template()` メソッドを使用してメッセージをフォーマットできます：

In [ ]:
messages = dataset[0]["messages"]
conversation = tokenizer.apply_chat_template(messages, tokenize=False)
print(conversation)

このチャットテンプレートは非常に洗練されているので、詳しく見てみましょう！まず、各メッセージの開始と終了を示す特別なトークン `<|start|>` と `<|end|>` があることがわかります。また、会話の終わりを示す `<|return|>` トークンもあります。これらのトークンは、モデルが会話の構造を理解するのに役立ちます。

また、_2つの_タイプのシステムメッセージがあることもわかります：

* すべてのメッセージに使用されるデフォルトの `system` メッセージ。上記の例では、これは _"You are ChatGPT, a large language model trained by OpenAI..."_ のテキストを指します
* カスタム指示を含む特別な `developer` メッセージ（`messages` オブジェクトの `system` ロールで定義）。これにより、特定の会話でモデルがどのように動作すべきかについて追加のコンテキストを提供できます。上記の例では、これは _"You are an AI chatbot with a lively and energetic personality."_ のテキストを指します

最後に、アシスタントの応答が一連の_チャンネル_に含まれていることがわかります：

* `analysis` チャンネルは、モデルがユーザーの質問について段階的に考えることができるモデルの推論プロセスに使用されます。上記の例では、これはフランス語のテキスト _"D'accord, l'utilisateur demande les tendances Twitter..."_ を指します
* `final` チャンネルは、ユーザーへのモデルの最終応答に使用されます。上記の例では、これは _"Hey there! While I can't check Twitter..."_ のテキストを指します

データセットの準備方法を理解したので、訓練用のモデルの準備に進みましょう。

## モデルの準備

訓練用にモデルを準備するために、まず[Hugging Face Hub](https://huggingface.co)から重みをダウンロードしましょう。Transformersの `AutoModelForCausalLM` クラスを使用してモデルを読み込みます：

In [ ]:
import torch
from transformers import AutoModelForCausalLM, Mxfp4Config

quantization_config = Mxfp4Config(dequantize=True)
model_kwargs = dict(
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    use_cache=False,
    device_map="auto",
)

model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs)

これにより、訓練に必要な設定でモデルが読み込まれます。`attn_implementation` はパフォーマンス向上のため `eager` に設定され、勾配チェックポイント付きでモデルをファインチューニングするため `use_cache` は `False` に設定されています。

Transformersに慣れている場合、量子化に `Mxfp4Config` を使用していることに気づくかもしれません。これは、AIワークロードに最適化された[MXFP4](https://en.wikipedia.org/wiki/Block_floating_point)と呼ばれる特別な4ビット浮動小数点フォーマットで混合精度訓練を可能にするOpenAIモデル専用の設定です。

モデルを訓練する前に、デフォルト設定でモデルがどのように動作するかを確認するため、サンプル応答を生成してみましょう。そのためには、サンプルプロンプトをトークン化し、モデルを使用して応答を生成する必要があります：

In [ ]:
messages = [
    {"role": "user", "content": "¿Cuál es el capital de Australia?"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(input_ids, max_new_tokens=512)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

この例では、モデルが最初に英語で質問について推論し、その後スペイン語で最終応答を提供することがわかります。これはモデルのデフォルト動作ですが、少しファインチューニングで変更できるかどうか見てみましょう。

そのために、[LoRA](https://huggingface.co/learn/llm-course/chapter11/4)（Low-Rank Adaptation）と呼ばれる技法を使用してモデルをファインチューニングします。この技法により、モデルの特定のレイヤーを調整できるため、`openai/gpt-oss-20b` のような大規模モデルに特に有用です。

まず、モデルを `PeftModel` としてラップし、LoRA設定を定義する必要があります。[PEFTライブラリ](https://github.com/huggingface/peft)の `LoraConfig` クラスを使用します：

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    target_parameters=[
        "7.mlp.experts.gate_up_proj",
        "7.mlp.experts.down_proj",
        "15.mlp.experts.gate_up_proj",
        "15.mlp.experts.down_proj",
        "23.mlp.experts.gate_up_proj",
        "23.mlp.experts.down_proj",
    ],
)
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

ここではLoRAの基本的なハイパーパラメータを使用しましたが、異なる値を試してモデルのパフォーマンスにどのような影響があるかを確認できます。例えば、`r` を増やすとより多くの訓練可能パラメータが有効になり、よりVRAMと訓練時間を必要とする代わりに、より良いモデルを生成する可能性があります。

**注意:** `openai/gpt-oss-20b` モデルは[Mixture-of-Experts (MoE)](https://huggingface.co/blog/moe)アーキテクチャです。アテンション層（`target_modules="all-linear"`）をターゲットにすることに加えて、エキスパートモジュール内の投影層も含めることが重要です。PEFTは `target_parameters` 引数を通じてこれを促進し、`mlp.experts.down_proj` や `mlp.experts.gate_up_proj` などのエキスパート固有の層を指定できます。この例では、これらの投影層のサブセットをターゲットにしていますが、異なる設定で実験することをお勧めします。

モデルとデータセットの準備ができたので、訓練用のハイパーパラメータを定義できます。

## ファインチューニング

TRLは、`SFTConfig` クラスを使用して訓練用のハイパーパラメータを定義する便利な方法を提供します。学習率、バッチサイズ、エポック数、その他のパラメータを以下のように設定します：

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    learning_rate=2e-4,
    gradient_checkpointing=True,
    num_train_epochs=1,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_length=2048,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    output_dir="gpt-oss-20b-multilingual-reasoner",
    report_to="trackio",
    push_to_hub=True,
)

`per_device_train_batch_size` は4に設定され、`gradient_accumulation_steps` は4に設定されていることに注意してください。これは、1つのGPUで実質的に4 x 4 = 16のバッチサイズを持つことを意味します。ハードウェア設定に基づいてこれらの値を調整する必要があるかもしれません。また、訓練の進捗とメトリクスをログに記録するために[Trackio](https://huggingface.co/blog/trackio)を使用していますが、選択に応じて他のロギングライブラリを使用できます。

これで、モデルを訓練するのに必要なすべての要素が揃いました。TRLの `SFTTrainer` クラスを使用して訓練プロセスを処理します。トレーナーは、データセットのフォーマット、チャットテンプレートの適用、モデルの訓練を処理します：

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

H100 GPUでは、これは約18分の訓練時間がかかりますが、ハードウェアによってはより長くかかる場合があります。

## モデルを保存し、Hugging Face Hubにプッシュ

最後に、ファインチューニングされたモデルをHubリポジトリにプッシュして、コミュニティと共有できます：

In [ ]:
trainer.save_model(training_args.output_dir)
trainer.push_to_hub(dataset_name="HuggingFaceH4/Multilingual-Thinking")

**注意**: メモリ不足（OOM）エラーを避けるため、この時点でカーネルを再起動することをお勧めします。訓練されたモデルはまだGPUメモリを占有していますが、もう必要ありません。

## 推論

モデルがHubにアップロードされると、推論に使用できます。そのために、まず元のベースモデルとそのトークナイザーを初期化します。次に、高速推論のためにファインチューニングされた重みをベースモデルとマージする必要があります：

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# トークナイザーを読み込み
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-20b")

# 最初に元のモデルを読み込み
model_kwargs = dict(attn_implementation="eager", torch_dtype="auto", use_cache=True, device_map="auto")
base_model = AutoModelForCausalLM.from_pretrained("openai/gpt-oss-20b", **model_kwargs).cuda()

# ファインチューニングされた重みをベースモデルとマージ
peft_model_id = "gpt-oss-20b-multilingual-reasoner"
model = PeftModel.from_pretrained(base_model, peft_model_id)
model = model.merge_and_unload()

モデルが読み込まれたので、最後のステップはそこからトークンを生成することです！ここでは、モデルの `generate` メソッドを使用して入力プロンプトに基づいて出力を生成します。まず、プロンプトを定義しましょう：

これで、プロンプトをトークン化し、出力を生成できます。最後に、出力トークンをデコードして最終応答を取得できます：

In [ ]:
REASONING_LANGUAGE = "German"
SYSTEM_PROMPT = f"reasoning language: {REASONING_LANGUAGE}"
USER_PROMPT = "¿Cuál es el capital de Australia?"  # Spanish for "What is the capital of Australia?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

gen_kwargs = {"max_new_tokens": 512, "do_sample": True, "temperature": 0.6, "top_p": None, "top_k": None}

output_ids = model.generate(input_ids, **gen_kwargs)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

中国語やヒンディー語など、モデルが明示的にファインチューニングされていない言語でも試してみましょう：

In [ ]:
REASONING_LANGUAGE = "Chinese"  # or Hindi, or any other language...
SYSTEM_PROMPT = f"reasoning language: {REASONING_LANGUAGE}"
USER_PROMPT = "What is the national symbol of Canada?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(input_ids, **gen_kwargs)
response = tokenizer.batch_decode(output_ids)[0]
print(response)

素晴らしい、動作します - `openai/gpt-oss-20b` を複数の言語で推論できるようにファインチューニングしました！

## 結論

おめでとうございます！TRLライブラリとLoRAを使用して多言語推論モデルのファインチューニングに成功しました。このノートブックの手順は、Hugging Face Hub上の他の多くの[データセット](https://huggingface.co/datasets)で [`openai/gpt-oss-20b`](https://huggingface.co/openai/gpt-oss-20b) をファインチューニングするために適応できます - あなたが何を構築するかを楽しみにしています！